In [ ]:
!pip install lmdb


In [ ]:
!apt-get update -qq && apt-get install -y ncbi-blast+

In [ ]:
# UniProt reviewed spider-toxin sequences (~1 300)
!wget -O spider_toxins.fasta \
"https://rest.uniprot.org/uniprotkb/search?query=(taxonomy_id:6893)+AND+(keyword:KW-0754)&format=fasta&reviewed=true"

In [ ]:
# Build the BLAST database (protein)
!makeblastdb -in spider_toxins.fasta -dbtype prot -out spider-toxins

In [ ]:
# quick sanity check
!which blastp
!ls spider-toxins.*

In [ ]:
def spider_neuro_aug(seq, mag):
    """
    Lightweight surrogate for the Spider-neuro homology filter.
    - 50 % subs (60 % conservative)
    - 10 % random insert
    - NO blastp call → instant
    """
    if not isinstance(mag, str):        # keep signature
        raise ValueError('spider_neuro_aug expects mag=db_path (ignored here)')

    seq = seq.copy()
    # 1. substitution
    for i in range(len(seq)):
        if random.random() < 0.5:
            seq[i] = _spider_similar_aa(seq[i]) if random.random() < 0.6 else _spider_random_aa()
    # 2. insertion
    if random.random() < 0.1:
        pos = random.randint(0, len(seq))
        seq[pos:pos] = _spider_random_aa()
    return seq



In [ ]:
results["spider_neuro"] = run_one_solubility("spider_neuro", spider_neuro_aug, mag="dummy")

In [ ]:
import os, lmdb, pickle, random, math, copy
import numpy as np
from tqdm import tqdm
from collections import OrderedDict
import subprocess as sp
from tempfile import NamedTemporaryFile
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.cross_decomposition import PLSRegression
from sklearn.neighbors import NearestNeighbors


# Initialize environment
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Protein constants
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
VOCAB = {aa: idx + 1 for idx, aa in enumerate(AMINO_ACIDS)}
VOCAB_SIZE = len(VOCAB) + 1  # 0 = padding
MAX_LEN = 512  # Maximum sequence length

def sequence_features(seq, max_len=MAX_LEN):
    # Simple features: amino acid counts, length, hydrophobic ratio, etc.
    features = [seq.count(aa) for aa in AMINO_ACIDS]
    features.append(len(seq))
    hydrophobic = set('AVLIMFWY')
    hydrophobic_count = sum(seq.count(aa) for aa in hydrophobic)
    features.append(hydrophobic_count / max(1, len(seq)))
    return np.array(features)

def load_lmdb_data(lmdb_path, max_len=MAX_LEN, transform=None, p=0.7, mag=0.15):
    env = lmdb.open(lmdb_path, readonly=True, lock=False)
    X, y = [], []
    with env.begin() as txn:
        for key, value in txn.cursor():
            try:
                rec = pickle.loads(value)
                if isinstance(rec, dict) and 'primary' in rec and 'solubility' in rec:
                    seq = list(rec['primary'])
                    label = float(rec['solubility'])
                    # Apply augmentation
                    if transform and random.random() < p:
                        try:
                            augmented = transform(seq, mag)
                            if len(augmented) >= 5:
                                seq = augmented
                        except Exception as e:
                            print(f"Augmentation error: {e}")
                            seq = list(rec['primary'])
                    seq_str = ''.join(seq[:max_len])
                    X.append(sequence_features(seq_str, max_len))
                    y.append(label)
            except Exception:
                continue
    print(f"Loaded {len(X)} valid samples from {lmdb_path}")
    return np.array(X), np.array(y)

class RandomForestSolubility:
    def __init__(self, n_estimators=100, random_state=SEED):
        self.model = RandomForestClassifier(n_estimators=n_estimators, random_state=random_state)

    def fit(self, X, y):
        self.model.fit(X, y)

    def predict(self, X):
        return self.model.predict(X)

    def predict_proba(self, X):
        return self.model.predict_proba(X)[:, 1]

# ------------------------------------------------------------
# Augmentation Functions
# ------------------------------------------------------------
CODON = {
    "TTT":"F","TTC":"F","TTA":"L","TTG":"L","CTT":"L","CTC":"L","CTA":"L","CTG":"L",
    "ATT":"I","ATC":"I","ATA":"I","ATG":"M","GTT":"V","GTC":"V","GTA":"V","GTG":"V",
    "TCT":"S","TCC":"S","TCA":"S","TCG":"S","AGT":"S","AGC":"S","CCT":"P","CCC":"P",
    "CCA":"P","CCG":"P","ACT":"T","ACC":"T","ACA":"T","ACG":"T","GCT":"A","GCC":"A",
    "GCA":"A","GCG":"A","TAT":"Y","TAC":"Y","TAA":"Stop","TAG":"Stop","CAT":"H",
    "CAC":"H","CAA":"Q","CAG":"Q","TGT":"C","TGC":"C","TGA":"Stop","TGG":"W",
    "CGT":"R","CGC":"R","CGA":"R","CGG":"R","AGA":"R","AGG":"R","GGT":"G",
    "GGC":"G","GGA":"G","GGG":"G",
}
AA2CODON = {aa:[c for c,a in CODON.items() if a==aa] for aa in set(CODON.values())}
STOP = {"TAA","TAG","TGA"}

# BLOSUM62 matrix (simplified for implementation)
BLOSUM62 = {
    'A': {'A': 4, 'R': -1, 'N': -2, 'D': -2, 'C': 0, 'Q': -1, 'E': -1, 'G': 0, 'H': -2, 'I': -1, 'L': -1, 'K': -1, 'M': -1, 'F': -2, 'P': -1, 'S': 1, 'T': 0, 'W': -3, 'Y': -2, 'V': 0},
    'R': {'A': -1, 'R': 5, 'N': 0, 'D': -2, 'C': -3, 'Q': 1, 'E': 0, 'G': -2, 'H': 0, 'I': -3, 'L': -2, 'K': 2, 'M': -1, 'F': -3, 'P': -2, 'S': -1, 'T': -1, 'W': -3, 'Y': -2, 'V': -3},
    'N': {'A': -2, 'R': 0, 'N': 6, 'D': 1, 'C': -3, 'Q': 0, 'E': 0, 'G': 0, 'H': 1, 'I': -3, 'L': -3, 'K': 0, 'M': -2, 'F': -3, 'P': -2, 'S': 1, 'T': 0, 'W': -4, 'Y': -2, 'V': -3},
    'D': {'A': -2, 'R': -2, 'N': 1, 'D': 6, 'C': -3, 'Q': 0, 'E': 2, 'G': -1, 'H': -1, 'I': -3, 'L': -4, 'K': -1, 'M': -3, 'F': -3, 'P': -1, 'S': 0, 'T': -1, 'W': -4, 'Y': -3, 'V': -3},
    'C': {'A': 0, 'R': -3, 'N': -3, 'D': -3, 'C': 9, 'Q': -3, 'E': -4, 'G': -3, 'H': -3, 'I': -1, 'L': -1, 'K': -3, 'M': -1, 'F': -2, 'P': -3, 'S': -1, 'T': -1, 'W': -2, 'Y': -2, 'V': -1},
    'Q': {'A': -1, 'R': 1, 'N': 0, 'D': 0, 'C': -3, 'Q': 5, 'E': 2, 'G': -2, 'H': 0, 'I': -3, 'L': -2, 'K': 1, 'M': 0, 'F': -3, 'P': -1, 'S': 0, 'T': -1, 'W': -2, 'Y': -1, 'V': -2},
    'E': {'A': -1, 'R': 0, 'N': 0, 'D': 2, 'C': -4, 'Q': 2, 'E': 5, 'G': -2, 'H': 0, 'I': -3, 'L': -3, 'K': 1, 'M': -2, 'F': -3, 'P': -1, 'S': 0, 'T': -1, 'W': -3, 'Y': -2, 'V': -2},
    'G': {'A': 0, 'R': -2, 'N': 0, 'D': -1, 'C': -3, 'Q': -2, 'E': -2, 'G': 6, 'H': -2, 'I': -4, 'L': -4, 'K': -2, 'M': -3, 'F': -3, 'P': -2, 'S': 0, 'T': -2, 'W': -2, 'Y': -3, 'V': -3},
    'H': {'A': -2, 'R': 0, 'N': 1, 'D': -1, 'C': -3, 'Q': 0, 'E': 0, 'G': -2, 'H': 8, 'I': -3, 'L': -3, 'K': -1, 'M': -2, 'F': -1, 'P': -2, 'S': -1, 'T': -2, 'W': -2, 'Y': 2, 'V': -3},
    'I': {'A': -1, 'R': -3, 'N': -3, 'D': -3, 'C': -1, 'Q': -3, 'E': -3, 'G': -4, 'H': -3, 'I': 4, 'L': 2, 'K': -3, 'M': 1, 'F': 0, 'P': -3, 'S': -2, 'T': -1, 'W': -3, 'Y': -1, 'V': 3},
    'L': {'A': -1, 'R': -2, 'N': -3, 'D': -4, 'C': -1, 'Q': -2, 'E': -3, 'G': -4, 'H': -3, 'I': 2, 'L': 4, 'K': -2, 'M': 2, 'F': 0, 'P': -3, 'S': -2, 'T': -1, 'W': -2, 'Y': -1, 'V': 1},
    'K': {'A': -1, 'R': 2, 'N': 0, 'D': -1, 'C': -3, 'Q': 1, 'E': 1, 'G': -2, 'H': -1, 'I': -3, 'L': -2, 'K': 5, 'M': -1, 'F': -3, 'P': -1, 'S': 0, 'T': -1, 'W': -3, 'Y': -2, 'V': -2},
    'M': {'A': -1, 'R': -1, 'N': -2, 'D': -3, 'C': -1, 'Q': 0, 'E': -2, 'G': -3, 'H': -2, 'I': 1, 'L': 2, 'K': -1, 'M': 5, 'F': 0, 'P': -2, 'S': -1, 'T': -1, 'W': -1, 'Y': -1, 'V': 1},
    'F': {'A': -2, 'R': -3, 'N': -3, 'D': -3, 'C': -2, 'Q': -3, 'E': -3, 'G': -3, 'H': -1, 'I': 0, 'L': 0, 'K': -3, 'M': 0, 'F': 6, 'P': -4, 'S': -2, 'T': -2, 'W': 1, 'Y': 3, 'V': -1},
    'P': {'A': -1, 'R': -2, 'N': -2, 'D': -1, 'C': -3, 'Q': -1, 'E': -1, 'G': -2, 'H': -2, 'I': -3, 'L': -3, 'K': -1, 'M': -2, 'F': -4, 'P': 7, 'S': -1, 'T': -1, 'W': -4, 'Y': -3, 'V': -2},
    'S': {'A': 1, 'R': -1, 'N': 1, 'D': 0, 'C': -1, 'Q': 0, 'E': 0, 'G': 0, 'H': -1, 'I': -2, 'L': -2, 'K': 0, 'M': -1, 'F': -2, 'P': -1, 'S': 4, 'T': 1, 'W': -3, 'Y': -2, 'V': -2},
    'T': {'A': 0, 'R': -1, 'N': 0, 'D': -1, 'C': -1, 'Q': -1, 'E': -1, 'G': -2, 'H': -2, 'I': -1, 'L': -1, 'K': -1, 'M': -1, 'F': -2, 'P': -1, 'S': 1, 'T': 5, 'W': -2, 'Y': -2, 'V': 0},
    'W': {'A': -3, 'R': -3, 'N': -4, 'D': -4, 'C': -2, 'Q': -2, 'E': -3, 'G': -2, 'H': -2, 'I': -3, 'L': -2, 'K': -3, 'M': -1, 'F': 1, 'P': -4, 'S': -3, 'T': -2, 'W': 11, 'Y': 2, 'V': -3},
    'Y': {'A': -2, 'R': -2, 'N': -2, 'D': -3, 'C': -2, 'Q': -1, 'E': -2, 'G': -3, 'H': 2, 'I': -1, 'L': -1, 'K': -2, 'M': -1, 'F': 3, 'P': -3, 'S': -2, 'T': -2, 'W': 2, 'Y': 7, 'V': -1},
    'V': {'A': 0, 'R': -3, 'N': -3, 'D': -3, 'C': -1, 'Q': -2, 'E': -2, 'G': -3, 'H': -3, 'I': 3, 'L': 1, 'K': -2, 'M': 1, 'F': -1, 'P': -2, 'S': -2, 'T': 0, 'W': -3, 'Y': -1, 'V': 4}
}

# phys-chem groups used in the paper
SPIDER_GROUPS = {
    'aliphatic': 'AVLIMC',
    'aromatic': 'FWYH',
    'polar': 'STNQ',
    'positive': 'KR',
    'negative': 'DE',
    'special': 'GP'
}
SPIDER_FLAT = list(''.join(SPIDER_GROUPS.values()))

def vanilla(seq, mag=None): return seq
def random_insert(seq, mag):
    for _ in range(max(1,int(mag*len(seq)))):
        seq.insert(random.randint(0,len(seq)), random.choice(AMINO_ACIDS))
    return seq
def random_delete(seq, mag):
    keep = [aa for aa in seq if random.random() > mag]
    return keep if keep else seq
def random_sub(seq, mag):
    seq = seq.copy()
    for i in range(len(seq)):
        if random.random() < mag:
            seq[i] = random.choice(AMINO_ACIDS)
    return seq
def random_swap(seq, mag):
    seq = seq.copy()
    for _ in range(max(1, int(mag*len(seq)))):
        i,j = random.sample(range(len(seq)), 2)
        seq[i], seq[j] = seq[j], seq[i]
    return seq
def crop(seq, mag):
    l = max(1, int(mag*len(seq)))
    start = random.randint(0, max(0, len(seq)-l))
    return seq[start:start+l]
def shuffle_seg(seq, mag):
    l = max(1, int(mag*len(seq)))
    start = random.randint(0, max(0, len(seq)-l))
    sub = seq[start:start+l]; random.shuffle(sub)
    return seq[:start] + sub + seq[start+l:]
def global_reverse(seq, mag=None): return seq[::-1]
def cut_shuffle(seq, mag):
    if len(seq) < 20: return seq
    k = max(2, int(mag*10))
    cuts = sorted(random.sample(range(1, len(seq)), min(k-1, len(seq)-1))) + [len(seq)]
    parts = [seq[i:j] for i,j in zip([0]+cuts[:-1], cuts)]
    random.shuffle(parts)
    return [aa for p in parts for aa in p]
def subsequence(seq, mag):
    if len(seq) < 20: return seq
    k = max(2, int(mag*10))
    # Split sequence into chunks of size k
    parts = [seq[i:i+k] for i in range(0, len(seq), k)]
    keep = max(1, int(mag*len(parts)))
    parts = random.sample(parts, keep)
    return [aa for p in parts for aa in p]

def _repeat_seq(seq):
    freq = {}
    for l in range(2, min(10, len(seq)//2+1)):
        for i in range(len(seq)-l+1):
            sub = "".join(seq[i:i+l])
            freq[sub] = freq.get(sub, 0) + 1
    return max(freq.items(), key=lambda x:x[1])[0] if freq and max(freq.values()) > 1 else None
def repeat_expansion(seq, mag):
    rep = _repeat_seq(seq)
    if rep is None: return seq
    L = len(rep); pos = [i for i in range(len(seq)-L+1) if "".join(seq[i:i+L])==rep]
    if not pos: return seq
    exp = random.sample(pos, max(1, int(mag*len(pos))))
    for p in sorted(exp, reverse=True): seq[p+L:p+L] = list(rep)
    return seq
def repeat_contraction(seq, mag):
    rep = _repeat_seq(seq)
    if rep is None: return seq
    L = len(rep); pos = [i for i in range(len(seq)-L+1) if "".join(seq[i:i+L])==rep]
    if not pos: return seq
    rem = random.sample(pos, max(1, int(mag*len(pos))))
    for p in sorted(rem, reverse=True): del seq[p:p+L]
    return seq

def back_translation(seq, mag):
    mRNA = []
    for aa in seq:
        if aa in AA2CODON:
            mRNA.extend(list(random.choice(AA2CODON[aa])))

    if not mRNA:
        return seq

    mRNA_len = len(mRNA)
    num_subs = max(0, int(mag * mRNA_len))
    for _ in range(num_subs):
        pos = random.randint(0, mRNA_len - 1)
        mRNA[pos] = random.choice(['A', 'U', 'C', 'G'])

    codons = ["".join(mRNA[i:i+3]) for i in range(0, len(mRNA), 3)]
    aa_seq = []
    for c in codons:
        if len(c) == 3:
            aa = CODON.get(c, 'X')
            if aa in AMINO_ACIDS:
                aa_seq.append(aa)

    return aa_seq

def blosum_substitute(seq, mag, temperature=1.0):
    sequence = seq.copy()
    for i in range(len(sequence)):
        if random.random() < mag:
            aa = sequence[i]
            if aa in BLOSUM62:
                weights = [math.exp(BLOSUM62[aa][other]/temperature) for other in AMINO_ACIDS]
                total = sum(weights)
                if total > 0:
                    probs = [w/total for w in weights]
                    new_aa = random.choices(AMINO_ACIDS, weights=probs, k=1)[0]
                    sequence[i] = new_aa
    return sequence

    X_train, y_train = load_lmdb_data(
        "/kaggle/input/solubility/solubility_train.lmdb",
        max_len=MAX_LEN,
        transform=transform,
        p=0.7,
        mag=mag,
    )
    X_val, y_val = load_lmdb_data("/kaggle/input/solubility/solubility_valid.lmdb", max_len=MAX_LEN)
    X_test, y_test = load_lmdb_data("/kaggle/input/solubility/solubility_test.lmdb", max_len=MAX_LEN)

    model = RandomForestSolubility()
    model.fit(X_train, y_train)
    val_pred = model.predict(X_val)
    val_acc = accuracy_score(y_val, val_pred)
    print(f"val_acc={val_acc:.4f}")
    test_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, test_pred)
    print(f"{aug_name:<18} test-acc: {test_acc:.4f}")
    return test_acc

def policy_wrapper(policy, seq, mag):
    """Apply augmentations according to policy probabilities"""
    aug_list = []
    for aug_name, prob in policy.items():
        if random.random() < prob:
            aug_list.append(aug_name)
   
    if not aug_list:
        return seq
   
    selected_aug = random.choice(aug_list)
    return AUGMENTATIONS[selected_aug](seq, mag)

def run_apa_search_solubility(n_policies=10, mag=0.15):
    print("\n>>> APA SEARCH FOR SOLUBILITY")
    X_train, y_train = load_lmdb_data(
        "/kaggle/input/solubility/solubility_train.lmdb",
        max_len=MAX_LEN,
        transform=lambda s, m: random.choice(list(AUGMENTATIONS.values()))(s, m),
        p=1.0,
        mag=mag
    )
    X_val, y_val = load_lmdb_data("/kaggle/input/solubility/solubility_valid.lmdb", max_len=MAX_LEN)

    best_acc, best_pol = 0.0, None
    for pid in range(n_policies):
        policy = {k: random.choice([0.0, 0.2, 0.4, 0.6, 0.8, 1.0]) for k in AUGMENTATIONS}
        X_train_pol, y_train_pol = load_lmdb_data(
            "/kaggle/input/solubility/solubility_train.lmdb",
            max_len=MAX_LEN,
            transform=lambda s, m: policy_wrapper(policy, s, m),
            p=1.0,
            mag=mag
        )
        model = RandomForestSolubility()
        model.fit(X_train_pol, y_train_pol)
        val_pred = model.predict(X_val)
        val_acc = accuracy_score(y_val, val_pred)
        if val_acc > best_acc:
            best_acc, best_pol = val_acc, policy
            print(f"New best policy: {val_acc:.4f}")

    # Final training with best policy
    X_test, y_test = load_lmdb_data("/kaggle/input/solubility/solubility_test.lmdb", max_len=MAX_LEN)
    X_train_best, y_train_best = load_lmdb_data(
        "/kaggle/input/solubility/solubility_train.lmdb",
        max_len=MAX_LEN,
        transform=lambda s, m: policy_wrapper(best_pol, s, m),
        p=1.0,
        mag=mag
    )
    model = RandomForestSolubility()
    model.fit(X_train_best, y_train_best)
    test_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, test_pred)
    print(f"APA (best)         test-acc: {test_acc:.4f}")
    return test_acc

# ------------------------------------------------------------
# Main Execution
# ------------------------------------------------------------
if __name__ == "__main__":
    # Define augmentations
    AUGMENTATIONS = OrderedDict([
        ("vanilla", vanilla),
        ("random_insert", random_insert),
        ("random_delete", random_delete),
        ("random_sub", random_sub),
        ("random_swap", random_swap),
        ("crop", crop),
        ("global_reverse", global_reverse),
        ("shuffle_seg", shuffle_seg),
        ("cut_shuffle", cut_shuffle),
        ("subsequence", subsequence),
        ("repeat_expansion", repeat_expansion),
        ("repeat_contraction", repeat_contraction),
        ("back_translation", back_translation),
        ("blosum_substitute", blosum_substitute),
    ])

    mag = 0.15
    results = OrderedDict()

    BLAST_DB_PATH = "/kaggle/working/spider-toxins"  # Change to your actual BLAST DB prefix path

    # Run all augmentations
    for name, fn in AUGMENTATIONS.items():
        if name == "spider_neuro":
            results[name] = run_one_solubility(name, fn, mag=BLAST_DB_PATH)
        else:
            results[name] = run_one_solubility(name, fn, mag=mag)

    # Run APA
    results["apa"] = run_apa_search_solubility(n_policies=10, mag=mag)

    # Print results
    print("\nFINAL SOLUBILITY RESULTS WITH RANDOM FOREST")
    print("-" * 42)
    for k, v in results.items():
        print(f"{k:<18}: {v:.4f}")
    print("-" * 42)